In [ ]:
# 1. Clone repository & system dependencies
%cd /content
!rm -rf /content/subtitlesssss
!git clone https://github.com/aananyaph/subtitlesssss
%cd /content/subtitlesssss

# Compatibility edits from original repository
!sed -i 's/^from typing import Optional$/from typing import Any, Callable, Dict, Optional/' main_app.py
!sed -i '0,/^import logging$/s//from __future__ import annotations\nimport logging/' app/loopback_capture.py

# Ensure system GUI/font libraries for offscreen Qt are installed
!apt-get install -y --no-install-recommends libgl1 libegl1 libxkbcommon0 libdbus-1-3 libfontconfig1
print("✅ System dependencies and repository ready.")


In [ ]:
# 2. Install required IndicTrans2, ONNX-GPU, and benchmarking packages
# Pre-install cython and wheel for IndicTransToolkit
!pip install --no-cache-dir wheel cython

# Install backend packages (note: faster-whisper pulls in CPU onnxruntime by default)
!pip install faster-whisper websockets jiwer scipy PyQt6 \
    sentencepiece sacremoses sacrebleu tokenizers indic-nlp-library pyngrok indictranstoolkit

# CRITICAL: faster-whisper installs CPU onnxruntime. We must swap it for onnxruntime-gpu:
!pip uninstall -y onnxruntime onnxruntime-gpu
!pip install onnxruntime-gpu
print("✅ All Python packages and onnxruntime-gpu successfully installed.")


In [ ]:
# 3. Verify CUDA, PyTorch, and ONNX Runtime GPU Providers
!nvidia-smi
import os
import sys
import types
import torch
from google.colab import userdata
# Compatibility fixes for Hugging Face transformers >= 4.45
# 1. Hugging Face removed transformers.onnx in newer releases, but IndicTrans2
#    remote code (configuration_indictrans.py) imports OnnxConfig, OnnxSeq2SeqConfigWithPast,
#    and compute_effective_axis_dimension from transformers.onnx and transformers.onnx.utils.
if 'transformers.onnx' not in sys.modules or not hasattr(sys.modules.get('transformers.onnx'), '__path__'):
    _mock_onnx = sys.modules.get('transformers.onnx') or types.ModuleType('transformers.onnx')
    _mock_onnx.__path__ = []
    class _OnnxConfig:
        default_fixed_batch = 2
        default_fixed_sequence = 16
    class _OnnxSeq2SeqConfigWithPast(_OnnxConfig): pass
    _mock_onnx.OnnxConfig = _OnnxConfig
    _mock_onnx.OnnxSeq2SeqConfigWithPast = _OnnxSeq2SeqConfigWithPast
    sys.modules['transformers.onnx'] = _mock_onnx
if 'transformers.onnx.utils' not in sys.modules:
    _mock_onnx_utils = types.ModuleType('transformers.onnx.utils')
    _mock_onnx_utils.compute_effective_axis_dimension = lambda dimension, fixed_dimension=None, num_token_to_add=0: ((fixed_dimension or 1) + (num_token_to_add or 0)) if (dimension == -1 or dimension is None) else dimension
    sys.modules['transformers.onnx.utils'] = _mock_onnx_utils
    setattr(sys.modules['transformers.onnx'], 'utils', _mock_onnx_utils)
try:
    import transformers
    setattr(transformers, 'onnx', sys.modules['transformers.onnx'])
except Exception:
    pass
# 2. IndicTransToolkit expects PreTrainedTokenizerBase in tokenization_utils
try:
    import transformers.tokenization_utils as _tu
    import transformers.tokenization_utils_base as _tub
    if not hasattr(_tu, 'PreTrainedTokenizerBase'):
        _tu.PreTrainedTokenizerBase = _tub.PreTrainedTokenizerBase
except Exception:
    pass
# 3. IndicTransTokenizer assigns special tokens before super().__init__(),
# causing AttributeError on _special_tokens_map in transformers >= 4.45.
try:
    from transformers.tokenization_utils_base import PreTrainedTokenizerBase
    if not getattr(PreTrainedTokenizerBase, '_patched_special_tokens', False):
        _orig_setattr = PreTrainedTokenizerBase.__setattr__
        def _safe_setattr(self, key, value):
            key_clean = key.removesuffix('_ids').removesuffix('_id') if key.endswith(('_id', '_ids')) else key
            if key_clean in getattr(self, 'SPECIAL_TOKENS_ATTRIBUTES', ()):
                if '_special_tokens_map' not in self.__dict__:
                    self.__dict__['_special_tokens_map'] = {}
            return _orig_setattr(self, key, value)
        PreTrainedTokenizerBase.__setattr__ = _safe_setattr
        PreTrainedTokenizerBase._patched_special_tokens = True
except Exception:
    pass
# 4. IndicTransForConditionalGeneration.tie_weights() defines def tie_weights(self):
# without **kwargs, but transformers >= 4.45 calls self.tie_weights(recompute_mapping=False).
try:
    import transformers.modeling_utils as _mu
    if not getattr(_mu.PreTrainedModel, '_patched_tie_weights', False):
        _orig_post_init = _mu.PreTrainedModel.post_init
        def _safe_post_init(self):
            cls = self.__class__
            if hasattr(cls, 'tie_weights'):
                orig_tw = cls.tie_weights
                if not getattr(orig_tw, '_safe_wrapped', False):
                    def _wrapped_tie_weights(self, *a, **kw):
                        try:
                            return orig_tw(self, *a, **kw)
                        except TypeError:
                            return orig_tw(self)
                    _wrapped_tie_weights._safe_wrapped = True
                    cls.tie_weights = _wrapped_tie_weights
            return _orig_post_init(self)
        _mu.PreTrainedModel.post_init = _safe_post_init
        _mu.PreTrainedModel._patched_tie_weights = True
except Exception:
    pass
try:
    import transformers.dynamic_module_utils as _dmu
    if not getattr(_dmu, '_patched_tie_weights', False):
        _orig_get_class = _dmu.get_class_from_dynamic_module
        def _safe_get_class(*args, **kwargs):
            cls = _orig_get_class(*args, **kwargs)
            if isinstance(cls, type) and hasattr(cls, 'tie_weights'):
                orig_tw = cls.tie_weights
                if not getattr(orig_tw, '_safe_wrapped', False):
                    def _wrapped_tie_weights(self, *a, **kw):
                        try:
                            return orig_tw(self, *a, **kw)
                        except TypeError:
                            return orig_tw(self)
                    _wrapped_tie_weights._safe_wrapped = True
                    cls.tie_weights = _wrapped_tie_weights
            return cls
        _dmu.get_class_from_dynamic_module = _safe_get_class
        _dmu._patched_tie_weights = True
except Exception:
    pass
# Hugging Face token from Colab secrets, environment, or prompt
try:
    _hf_token = userdata.get('HF_TOKEN')
    if _hf_token:
        os.environ['HF_TOKEN'] = _hf_token.strip()
        print('✅ Successfully loaded HF_TOKEN from Colab Secrets.')
except Exception:
    pass
if not os.environ.get('HF_TOKEN'):
    _manual_token = input("Paste your Hugging Face Access Token (from https://huggingface.co/settings/tokens) or press Enter to skip: ").strip() or None
    if _manual_token:
        os.environ['HF_TOKEN'] = _manual_token
if os.environ.get('HF_TOKEN'):
    try:
        from huggingface_hub import login
        login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
        print('✅ Hugging Face authentication verified and active.')
    except Exception as e:
        print('⚠️ Hugging Face login notice:', e)
else:
    print("⚠️ Warning: HF_TOKEN not provided. Gated AI4Bharat models require an authorized Hugging Face token.")
# Assert CUDA availability for PyTorch
print('PyTorch version:', torch.__version__)
assert torch.cuda.is_available(), 'CUDA is unavailable in PyTorch! Switch to T4 GPU.'
print('✅ PyTorch CUDA GPU:', torch.cuda.get_device_name(0))
import onnxruntime as ort
print('ONNX Runtime available providers:', ort.get_available_providers())
assert 'CUDAExecutionProvider' in ort.get_available_providers(), 'CUDAExecutionProvider unavailable!'
print('✅ ONNX Runtime CUDA provider verified.')
from IndicTransToolkit import IndicProcessor
ip = IndicProcessor(inference=True)
print('✅ IndicProcessor successfully initialized.')


In [ ]:
# 4. Pre-download IndicTrans2 1B repos and IndicConformer
import os, shutil
from huggingface_hub import snapshot_download, login

hf_token = os.environ.get('HF_TOKEN') or None
if hf_token:
    try:
        login(token=hf_token, add_to_git_credential=False)
    except Exception:
        pass

# Pre-download IndicConformer for Indic ASR
try:
    print('Downloading IndicConformer ASR model (ai4bharat/indic-conformer-600m-multilingual)...')
    snapshot_download('ai4bharat/indic-conformer-600m-multilingual', token=hf_token)
    print('✅ IndicConformer ready.')
except Exception as e:
    print(f'⚠️ Could not download IndicConformer: {e}')
    print('👉 If this model is gated, accept terms at: https://huggingface.co/ai4bharat/indic-conformer-600m-multilingual')

# Pre-download IndicTrans2 1B translation models
repos_1b = [
    'ai4bharat/indictrans2-en-indic-1B',
    'ai4bharat/indictrans2-indic-en-1B',
    'ai4bharat/indictrans2-indic-indic-1B',
]

for repo in repos_1b:
    try:
        print(f'Pre-downloading/verifying: {repo}...')
        snapshot_download(repo, token=hf_token)
        print(f'✅ Ready: {repo}')
    except Exception as e:
        print(f'⚠️ Could not download {repo}: {e}')
        print(f'👉 If this model is gated, accept terms at: https://huggingface.co/{repo}')


In [ ]:
# 5. Apply dynamic ASR config patch and extract ONNX fallback translator
import os

path = '/content/subtitlesssss/backend/app_bridge.py'
with open(path, 'r', encoding='utf-8') as f:
    s = f.read()

if 'def apply_asr_config' not in s:
    helper = '''# ---------- Dynamic ASR config (from client HANDSHAKE) ----------
_current_cfg_key = None

def apply_asr_config(cfg):
    """Rebuild pipeline to match client's audio/subtitle language choice."""
    global pipeline, _current_cfg_key
    key = (
        cfg.get("model_name") or "base",
        cfg.get("indic_lang") or None,
        cfg.get("asr_language") or None,
        cfg.get("asr_task") or None,
    )
    if pipeline is not None and key == _current_cfg_key:
        logging.info(f"[Bridge] ASR config unchanged {key}; keeping pipeline")
        return
    logging.info(f"[Bridge] Switching ASR config -> model={key[0]} indic={key[1]} lang={key[2]} task={key[3]}")
    import main_app
    old = pipeline
    if old is not None:
        try:
            old.stop()
        except Exception as e:
            logging.warning(f"[Bridge] old pipeline stop failed: {e}")
    if key[1]:
        from indic_backend import load_indic_model
        load_indic_model()
    new = main_app.DesktopLiveSubtitlePipeline(
        overlay=BridgeOverlay(),
        model_name=key[0],
        indic_lang=key[1],
        asr_language=key[2],
        asr_task=key[3],
    )
    try:
        new.start()
    except Exception as e:
        logging.warning(f"[Bridge] new pipeline start failed (no audio device?) {e}")
    pipeline = new
    _current_cfg_key = key
    logging.info("[Bridge] ASR config applied")

'''
    marker = '# ---------- WebSocket handler ----------'
    s = s.replace(marker, helper + marker)
    old = '                    logging.info(f"[Bridge] HANDSHAKE {data}")\n'
    new = '''                    logging.info(f"[Bridge] HANDSHAKE {data}")
                    _cfg = data.get("asrConfig")
                    if _cfg:
                        try:
                            await asyncio.get_running_loop().run_in_executor(None, apply_asr_config, _cfg)
                        except Exception as e:
                            logging.error(f"[Bridge] apply_asr_config failed: {e}", exc_info=True)
                            await ws.send(json.dumps({"type": "final", "text": f"Server model load failed: {e}", "is_final": True}))
'''
    s = s.replace(old, new)
    with open(path, 'w', encoding='utf-8') as f:
        f.write(s)
    print('app_bridge.py patched: dynamic ASR config on HANDSHAKE')

# Extract translator_live.py (ONNX fallback) if needed
root = '/content/subtitlesssss'
if not os.path.exists(f'{root}/translator_live.py'):
    import ast
    src = open(f'{root}/pipeline.py', encoding='utf-8').read()
    lines = src.splitlines(keepends=True)
    chunks = []
    for node in ast.parse(src).body:
        if isinstance(node, ast.Assign) and any(getattr(t, 'id', None) == 'LANG_MAP' for t in node.targets):
            chunks.append(''.join(lines[node.lineno - 1:node.end_lineno]))
        elif isinstance(node, ast.ClassDef) and node.name == 'IndicTransONNXModel':
            chunks.append(''.join(lines[node.lineno - 1:node.end_lineno]))
    header = '''import json, logging
from pathlib import Path
import numpy as np
import onnxruntime as ort
from huggingface_hub import snapshot_download
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
from indicnlp.tokenize import indic_detokenize
from indicnlp.transliterate.unicode_transliterate import UnicodeIndicTransliterator
from sacremoses import MosesDetokenizer, MosesTokenizer
from tokenizers import Tokenizer

logger = logging.getLogger("translator_live")

'''
    open(f'{root}/translator_live.py', 'w', encoding='utf-8').write(header + ''.join(chunks))
    print('translator_live.py extracted and ready.')


In [ ]:
# 6. Direct Model Benchmark (6 Directions, 20 Sentences Each)
!cd /content/subtitlesssss && git pull origin main

import importlib
import gc
import json
import os
import sys
import time
import numpy as np

sys.path.insert(0, '/content/subtitlesssss')

if 'translator_gpu' in sys.modules:
    importlib.reload(sys.modules['translator_gpu'])
if 'translation_proxy' in sys.modules:
    importlib.reload(sys.modules['translation_proxy'])

import sacrebleu
import torch
import translation_proxy as tp

# 20 parallel test sentences across English, Hindi, and Kannada
test_sentences = {
    'en': [
        'Hello, how are you today?',
        'Good morning to everyone present here.',
        'The weather is very pleasant this evening.',
        'Where is the nearest railway station?',
        'Can you please help me with this problem?',
        'We are developing a real-time subtitle translation system.',
        'Artificial intelligence is transforming communication across the world.',
        'I enjoy listening to classical Indian music while working.',
        'Please make sure to submit your assignment before Friday.',
        'The train will arrive at platform number three in ten minutes.',
        'India is a country with rich cultural heritage and diversity.',
        'What time does the library close on weekends?',
        'Health and education are fundamental rights for every citizen.',
        'She works as a software engineer in Bengaluru.',
        'Can you recommend a good book to read during vacation?',
        'The scientific conference starts tomorrow at nine in the morning.',
        'Yoga and meditation help maintain both mental and physical health.',
        'Renewable energy sources are essential for sustainable growth.',
        'Thank you very much for your kind support and cooperation.',
        'Have a wonderful day ahead and stay safe.'
    ],
    'hi': [
        'नमस्ते, आप आज कैसे हैं?',
        'यहाँ उपस्थित सभी लोगों को शुभ प्रभात।',
        'आज शाम का मौसम बहुत सुहावना है।',
        'निकटतम रेलवे स्टेशन कहाँ है?',
        'क्या आप कृपया इस समस्या में मेरी मदद कर सकते हैं?',
        'हम एक रीयल-टाइम उपशीर्षक अनुवाद प्रणाली विकसित कर रहे हैं।',
        'कृत्रिम बुद्धिमत्ता दुनिया भर में संचार को बदल रही है।',
        'मुझे काम करते समय भारतीय शास्त्रीय संगीत सुनना पसंद है।',
        'कृपया शुक्रवार से पहले अपना असाइनमेंट जमा करना सुनिश्चित करें।',
        'ट्रेन दस मिनट में प्लेटफॉर्म नंबर तीन पर पहुंचेगी।',
        'भारत समृद्ध सांस्कृतिक विरासत और विविधता वाला देश है।',
        'सप्ताहांत पर पुस्तकालय कितने बजे बंद होता है?',
        'स्वास्थ्य और शिक्षा प्रत्येक नागरिक के मौलिक अधिकार हैं।',
        'वह बेंगलुरु में एक सॉफ्टवेयर इंजीनियर के रूप में काम करती है।',
        'क्या आप छुट्टियों के दौरान पढ़ने के लिए किसी अच्छी किताब की सिफारिश कर सकते हैं?',
        'वैज्ञानिक सम्मेलन कल सुबह नौ बजे शुरू होगा।',
        'योग और ध्यान मानसिक और शारीरिक स्वास्थ्य दोनों को बनाए रखने में मदद करते हैं।',
        'सतत विकास के लिए नवीकरणीय ऊर्जा स्रोत आवश्यक हैं।',
        'आपके सहयोग और समर्थन के लिए बहुत-बहुत धन्यवाद।',
        'आपका दिन मंगलमय हो और सुरक्षित रहें।'
    ],
    'kn': [
        'ನಮಸ್ಕಾರ, ನೀವು ಇಂದು ಹೇಗಿದ್ದೀರಿ?',
        'ಇಲ್ಲಿ ಉಪಸ್ಥಿತರಿರುವ ಎಲ್ಲರಿಗೂ ಶುಭೋದಯ.',
        'ಇಂದು ಸಂಜೆಯ ಹವಾಮಾನವು ತುಂಬಾ ಆಹ್ಲಾದಕರವಾಗಿದೆ.',
        'ಹತ್ತಿರದ ರೈಲ್ವೆ ನಿಲ್ದಾಣ ಎಲ್ಲಿದೆ?',
        'ದಯವಿಟ್ಟು ಈ ಸಮಸ್ಯೆಯನ್ನು ಬಗೆಹರಿಸಲು ನನಗೆ ಸಹಾಯ ಮಾಡಬಹುದೇ?',
        'ನಾವು ನೈಜ-ಸಮಯದ ಉಪಶೀರ್ಷಿಕೆ ಅನುವಾದ ವ್ಯವಸ್ಥೆಯನ್ನು ಅಭಿವೃದ್ಧಿಪಡಿಸುತ್ತಿದ್ದೇವೆ.',
        'ಕೃತಕ ಬುದ್ಧಿಮತ್ತೆಯು ಪ್ರಪಂಚದಾದ್ಯಂತ ಸಂವಹನವನ್ನು ಪರಿವರ್ತಿಸುತ್ತಿದೆ.',
        'ಕೆಲಸ ಮಾಡುವಾಗ ನನಗೆ ಭಾರತೀಯ ಶಾಸ್ತ್ರೀಯ ಸಂಗೀತವನ್ನು ಕೇಳಲು ಇಷ್ಟ.',
        'ದಯವಿಟ್ಟು ಶುಕ್ರವಾರದ ಮೊದಲು ನಿಮ್ಮ ಕಾರ್ಯಯೋಜನೆಯನ್ನು ಸಲ್ಲಿಸುವುದನ್ನು ಖಚಿತಪಡಿಸಿಕೊಳ್ಳಿ.',
        'ರೈಲು ಹತ್ತು ನಿಮಿಷಗಳಲ್ಲಿ ಮೂರನೇ ಪ್ಲಾಟ್‌ಫಾರ್ಮ್‌ಗೆ ಆಗಮಿಸಲಿದೆ.',
        'ಭಾರತವು ಶ್ರೀಮಂತ ಸಾಂಸ್ಕೃತಿಕ ಪರಂಪರೆ ಮತ್ತು ವೈವಿಧ್ಯತೆಯನ್ನು ಹೊಂದಿರುವ ದೇಶವಾಗಿದೆ.',
        'ವಾರಾಂತ್ಯದಲ್ಲಿ ಗ್ರಂಥಾಲಯವು ಯಾವ ಸಮಯಕ್ಕೆ ಮುಚ್ಚುತ್ತದೆ?',
        'ಆರೋಗ್ಯ ಮತ್ತು ಶಿಕ್ಷಣ ಪ್ರತಿಯೊಬ್ಬ ನಾಗರಿಕನ ಮೂಲಭೂತ ಹಕ್ಕುಗಳು.',
        'ಅವಳು ಬೆಂಗಳೂರಿನಲ್ಲಿ ಸಾಫ್ಟ್‌ವೇರ್ ಎಂಜಿನಿಯರ್ ಆಗಿ ಕೆಲಸ ಮಾಡುತ್ತಿದ್ದಾಳೆ.',
        'ರಜಾದಿನಗಳಲ್ಲಿ ಓದಲು ಉತ್ತಮ ಪುಸ್ತಕವನ್ನು ನೀವು ಶಿಫಾರಸು ಮಾಡಬಹುದೇ?',
        'ವೈಜ್ಞಾನಿಕ ಸಮ್ಮೇಳನವು ನಾಳೆ ಬೆಳಿಗ್ಗೆ ಒಂಬತ್ತು ಗಂಟೆಗೆ ಪ್ರಾರಂಭವಾಗುತ್ತದೆ.',
        'ಯೋಗ ಮತ್ತು ಧ್ಯಾನ ಮಾನಸಿಕ ಮತ್ತು ದೈಹಿಕ ಆರೋಗ್ಯವನ್ನು ಕಾಪಾಡಿಕೊಳ್ಳಲು ಸಹಾಯ ಮಾಡುತ್ತವೆ.',
        'ಸುಸ್ಥಿರ ಬೆಳವಣಿಗೆಗೆ ನವೀಕರಿಸಬಹುದಾದ ಇಂಧನ ಮೂಲಗಳು ಅತ್ಯಗತ್ಯ.',
        'ನಿಮ್ಮ ಪ್ರೀತಿಯ ಬೆಂಬಲ ಮತ್ತು ಸಹಕಾರಕ್ಕೆ ತುಂಬಾ ಧನ್ಯವಾದಗಳು.',
        'ನಿಮ್ಮ ಮುಂದಿನ ದಿನವು ಅದ್ಭುತವಾಗಿರಲಿ ಮತ್ತು ಸುರಕ್ಷಿತವಾಗಿರಿ.'
    ]
}

directions = [('en', 'hi'), ('hi', 'en'), ('en', 'kn'), ('kn', 'en'), ('hi', 'kn'), ('kn', 'hi')]
configs = [
    ('torch1b (beam 2)', 'torch1b', '2'),
    ('torch1b (greedy)', 'torch1b', '1'),
    ('onnx (int8 distilled)', 'onnx', '1'),
]

benchmark_results = {}
print('==========================================================================')
print('BENCHMARK: 20 sentences per direction, 6 directions')
print('Comparing: torch1b beam 2, torch1b greedy, onnx int8 distilled')
print('==========================================================================\n')

for cfg_name, backend_mode, beam_val in configs:
    os.environ['TRANSLATOR_BACKEND'] = backend_mode
    os.environ['TR_BEAMS'] = beam_val
    tp.TRANSLATOR_BACKEND = backend_mode

    with tp._tr_lock:
        tp._model_cache.clear()
        tp._text_cache.clear()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

    print(f'\n>>> RUNNING CONFIG: {cfg_name} (backend={backend_mode}, beams={beam_val})')
    benchmark_results[cfg_name] = {}

    for src_iso, tgt_iso in directions:
        dir_key = f'{src_iso}->{tgt_iso}'
        sentences = test_sentences[src_iso]
        references = test_sentences[tgt_iso]
        latencies = []
        outputs = []

        # Warmup
        try:
            tp.translate_with_meta('Hello test.', src_iso, tgt_iso)
        except Exception:
            pass

        for s in sentences:
            t0 = time.perf_counter()
            out, _, _ = tp.translate_with_meta(s, src_iso, tgt_iso)
            latencies.append((time.perf_counter() - t0) * 1000.0)
            outputs.append(out)

        p50 = float(np.percentile(latencies, 50))
        p95 = float(np.percentile(latencies, 95))
        peak_vram_mb = (torch.cuda.max_memory_allocated() / (1024.0 * 1024.0)) if torch.cuda.is_available() else 0.0
        score_chrf = round(sacrebleu.corpus_chrf(outputs, [[r] for r in references], word_order=2).score, 2)

        benchmark_results[cfg_name][dir_key] = {
            'p50_ms': round(p50, 1),
            'p95_ms': round(p95, 1),
            'peak_vram_mb': round(peak_vram_mb, 1),
            'chrf2': score_chrf,
        }
        print(f'  [{dir_key}] p50: {p50:.1f}ms | p95: {p95:.1f}ms | Peak VRAM: {peak_vram_mb:.1f}MB | chrF++: {score_chrf}')

print('\n✅ Benchmark completed successfully!')


In [ ]:
# 7. Start ASR Bridge (8766) & Translation Proxy (8765)
import os
import time
import socket
import subprocess

os.makedirs('/content/subtitlesssss', exist_ok=True)

# 1. Cleanly kill any previous bridge or proxy instances
os.system('fuser -k -9 8765/tcp 8766/tcp 2>/dev/null || true')
os.system("pkill -9 -f 'backend/app_bridge.py' || true")
os.system("pkill -9 -f 'translation_proxy.py' || true")
time.sleep(2)

# 2. Launch app_bridge.py (ASR upstream on 8766)
bridge_log = open('/content/subtitlesssss/server.log', 'w')
bridge_proc = subprocess.Popen(
    ['python', 'backend/app_bridge.py', '--host', '127.0.0.1', '--model', 'distil-small.en', '--port', '8766'],
    cwd='/content/subtitlesssss',
    env=dict(os.environ, QT_QPA_PLATFORM='offscreen'),
    stdout=bridge_log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)
print(f'Launched app_bridge.py (PID: {bridge_proc.pid}). Waiting 15s for ASR initialization...', flush=True)
for rem in (10, 5, 0):
    time.sleep(5)
    if rem > 0:
        print(f'  Initializing ASR... ({rem}s remaining)', flush=True)

# 3. Launch translation_proxy.py (GPU Translation proxy on 8765)
proxy_log = open('/content/subtitlesssss/proxy.log', 'w')
proxy_proc = subprocess.Popen(
    ['python', 'translation_proxy.py'],
    cwd='/content/subtitlesssss',
    env=dict(os.environ, PROXY_PORT='8765', UPSTREAM_URL='ws://127.0.0.1:8766', TRANSLATOR_BACKEND='torch1b'),
    stdout=proxy_log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)
print(f'Launched translation_proxy.py (PID: {proxy_proc.pid}). Waiting 5s...', flush=True)
time.sleep(5)

# 4. Verify Listening Ports via Python Sockets
def is_port_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(2.0)
        return s.connect_ex(('127.0.0.1', port)) == 0

bridge_ok = is_port_open(8766)
proxy_ok = is_port_open(8765)

print('\n--- Server Status Check ---')
print(f'Port 8766 (ASR Bridge):      {"✅ LISTENING" if bridge_ok else "⚠️ NOT READY"}')
print(f'Port 8765 (Translation Proxy): {"✅ LISTENING" if proxy_ok else "⚠️ NOT READY"}')

if not bridge_ok:
    print('\n⚠️ Warning: app_bridge.py is not listening. Recent server.log:')
    os.system('tail -n 12 /content/subtitlesssss/server.log')
if not proxy_ok:
    print('\n⚠️ Warning: translation_proxy.py is not listening. Recent proxy.log:')
    os.system('tail -n 12 /content/subtitlesssss/proxy.log')

print('\n--- GPU Memory Usage ---')
os.system('nvidia-smi --query-gpu=memory.used,memory.total,memory.free --format=csv')


In [ ]:
# 8. Start Pyngrok Tunnel (8765)
import os
import time
from pyngrok import conf, ngrok

# 1. Cleanly terminate any previous ngrok instances
os.system('pkill -9 -f ngrok 2>/dev/null || true')
try:
    ngrok.kill()
except Exception:
    pass
time.sleep(1)

# 2. Load NGROK_AUTHTOKEN from Colab Secrets, environment, or prompt
token = None
try:
    from google.colab import userdata
    token = userdata.get('NGROK_AUTHTOKEN')
    if token:
        print('✅ Loaded NGROK_AUTHTOKEN from Colab Secrets.')
except Exception:
    pass

if not token:
    token = os.environ.get('NGROK_AUTHTOKEN')

if not token:
    print('⚠️ NGROK_AUTHTOKEN was not found in Colab Secrets (the 🔑 icon on the left panel).')
    token = input('Paste your ngrok authtoken (from https://dashboard.ngrok.com/get-started/your-authtoken): ').strip() or None

# 3. Configure authtoken
if token:
    os.environ['NGROK_AUTHTOKEN'] = token
    conf.get_default().auth_token = token
    ngrok.set_auth_token(token)
    print('✅ Ngrok authtoken configured successfully.')

# 4. Connect tunnel using pyngrok Python API
domain = 'aspire-reluctant-obsession.ngrok-free.dev'
print(f'Connecting ngrok tunnel to proxy port 8765 (attempting static domain "{domain}")...')

tunnel = None
try:
    tunnel = ngrok.connect(8765, 'http', domain=domain)
except Exception as e:
    print(f'ℹ️ Note: Static domain "{domain}" could not be bound ({e}).')
    print('Connecting with standard free dynamic ngrok tunnel...')

if tunnel is None:
    try:
        tunnel = ngrok.connect(8765, 'http')
    except Exception as e2:
        print(f'❌ Failed to start ngrok tunnel: {e2}')
        raise

public_url = tunnel.public_url
wss_url = public_url.replace('https://', 'wss://').replace('http://', 'ws://')

print('\n========================================================')
print('✅ NGROK IS ONLINE!')
print('Public HTTPS URL:     ', public_url)
print('Client WebSocket URL: ', wss_url)
print('========================================================')
if domain not in public_url:
    print(f'👉 Copy & paste this Client WebSocket URL into your PC Control Panel:\n   {wss_url}')
print('========================================================\n')


In [ ]:
# 9. Verify GPU Translation Layer & Latency
import os
import sys
import time

sys.path.insert(0, '/content/subtitlesssss')
import translation_proxy as tp

print('\n--- Testing Direct GPU Translation through Proxy Layer ---')
test_cases = [
    ('hi', 'en', 'नमस्ते, आप आज कैसे हैं?'),
    ('en', 'hi', 'Welcome to real-time subtitle translation.'),
    ('kn', 'en', 'ನಮಸ್ಕಾರ, ನೀವು ಇಂದು ಹೇಗಿದ್ದೀರಿ?'),
]

for src, tgt, sample in test_cases:
    t0 = time.perf_counter()
    out, backend, cache_hit = tp.translate_with_meta(sample, src, tgt)
    dt_ms = (time.perf_counter() - t0) * 1000
    print(f'[{src} -> {tgt}] ({backend}) in {dt_ms:.1f}ms')
    print(f'   Input:  {sample}')
    print(f'   Output: {out}\n')

print('========================================================')
print('✅ GPU TRANSLATION LAYER FULLY VERIFIED AND WORKING!')
print('========================================================')


In [ ]:
# 10. Live Logs Monitor
!grep 'SUB \|translation layer\|new upstream\|error' /content/subtitlesssss/proxy.log | tail -n 25
